Deepseek API key: YOUR_API_KEY

In [2]:
from openai import OpenAI
from typing import List, Dict, Any
from const import AI_CLASSS
import os
from utils import extract_entities, save_json_as_csv, load_json, get_ners, save_json

LLMs Agentic Workflow for NER (Prompt-Chaining):

Decomposed the entity extraction tasks to:
1. Mention Detection
2. Entity Typing

Using two LLMs call to do the tasks

In [3]:
def llm_call(prompt: str, system_prompt: str = "", model="deepseek-chat") -> str:
    """
    Calls the model with the given prompt and returns the response.

    Args:
        prompt (str): The user prompt to send to the model.
        system_prompt (str, optional): The system prompt to send to the model. Defaults to "".
        model (str, optional): The model to use for the call. Defaults to "claude-3-5-sonnet-20241022".

    Returns:
        str: The response from the language model.
    """
    client = OpenAI(api_key="YOUR_API_KEY", base_url="https://api.deepseek.com") # TODO: Change the API key
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=4096,
        stream=False
    )
    return response.choices[0].message.content

In [4]:
def chain(input: str, prompts: List[str]) -> str:
    """Chain multiple LLM calls sequentially, passing results between steps."""
    response = input
    result = []
    result.append(response)
    for i, prompt in enumerate(prompts, 1):
        print(response)
        response = llm_call(f"{prompt}\nInput: {response}")
        result.append(response)
    return result

In [ ]:
one_stage_prompt = """Extract all entities from the text below and output them in JSON format. Each entity should include:
- text: The entity mention
- type: Entity type. The interested types are: 'project', 'location', 'model', 'experiment', 'platform', 'instrument', 'provider', 'variable', 'weather event', 'natural hazard', 'teleconnection', 'ocean circulation'.

Note: If the same entity appears multiple times in the text, include it multiple times in the results.
"""

doc_files = os.listdir('./datasets/climate/')
data = load_json(f'./datasets/climate/{doc_files[0]}')
text = data[0]['text']
output = llm_call(f"{one_stage_prompt}\n\nInput: {text}")
print(output)

## ClimateNER

基础Workflow:

Workflow to improve the LLMs NER performance
- Mention Detection
    - CoT
    - Generation Format:
        - Has Negative: Directly generate entity mention span
        - No Negative: Using special tag, <e>Mention Span<e>
- Entity Typing

In [5]:
zero_shot_data_processing_steps = [
    """You are a mention detection assistant. Your task is to identify named entities in a given text and wrap each entity mention with <entity> and </entity> tags. 

    Instructions:
    1. Return the *exact same text*, but with each named entity enclosed in <entity>...</entity> tags.
    2. Named entities are from Climate domain, which includes the following types: 'project', 'location', 'model', 'experiment', 'platform', 'instrument', 'provider', 'variable', 'weather event', 'natural hazard', 'teleconnection', 'ocean circulation'.
    3. If there are multiple words in an entity (e.g., "the temperature is"), wrap the entire phrase: "the <entity>temperature</entity> is".
    4. Keep the rest of the text unchanged.
    5. Do not add any extra commentary or formatting. Only return the modified text.
    6. If there are no entities, simply return the original text without any tags.""",
    
    """You are an entity typing assistant. You will receive text where certain substrings are wrapped in <entity>...</entity> tags. These substrings are entity mentions. 

    Your task:
    1. For each mention wrapped in <entity>...</entity>, determine the most suitable entity type from the following list:
    - project
    - location
    - model
    - experiment
    - platform
    - instrument
    - provider
    - variable
    - weather event
    - natural hazard
    - teleconnection
    - ocean circulation
    - other  (use this if the mention does not fit any of the above)

    2. Convert each <entity>...</entity> tag into <entity type="TYPE">...</entity>, where TYPE is one of the listed options. 
    - Example: If the mention is "temperature" and you decide it’s an 'variable', change <entity>temperature</entity> to <entity type="variable">temperature</entity>.

    3. Leave all other text (outside the entity tags) exactly the same.

    4. If a mention is ambiguous or does not clearly fit any of the predefined types, classify it as "other".

    5. Output only the modified text, with the <entity> tags updated to include the type. Do not add any extra commentary, code fences, or explanation."""
]

In [14]:
doc_files = os.listdir('./datasets/climate/')
data = load_json(f'./datasets/climate/{doc_files[0]}')
text = data[0]['text']
output = chain(text, zero_shot_data_processing_steps)
ner_predicts = extract_entities(output[-1])

<heading>ABSTRACT</heading>
Future energy demand is likely to increase due to climate change, but the magnitude depends on many interacting sources of uncertainty. We combine econometrically estimated responses of energy use to income, hot and cold days with future projections of spatial population and national income under five socioeconomic scenarios and temperature increases around 2050 for two emission scenarios simulated by 21 Earth System Models (ESMs). Here we show that, across 210 realizations of socioeconomic and climate scenarios, vigorous (moderate) warming increases global climate-exposed energy demand before adaptation around 2050 by 25-58% (11-27%), on top of a factor 1.7-2.8 increase above present-day due to socioeconomic developments. We find broad agreement among ESMs that energy demand rises by more than 25% in the tropics and southern regions of the USA, Europe and China. Socioeconomic scenarios vary widely in the number of people in lowincome countries exposed to in

In [16]:
ner_predicts

[['socioeconomic scenarios', 'other'],
 ['emission scenarios', 'other'],
 ['Earth System Models', 'model'],
 ['ESMs', 'model'],
 ['socioeconomic', 'other'],
 ['climate scenarios', 'other'],
 ['climate-exposed energy demand', 'other'],
 ['socioeconomic developments', 'other'],
 ['ESMs', 'model'],
 ['tropics', 'location'],
 ['USA', 'location'],
 ['Europe', 'location'],
 ['China', 'location'],
 ['Socioeconomic scenarios', 'other'],
 ['low-income countries', 'location'],
 ['hot season cooling demand', 'other'],
 ['cold season heating demand', 'other'],
 ['economic sectors', 'other'],
 ['economic sectors', 'other'],
 ['irrigation', 'other'],
 ['crop-growing seasons', 'other']]

In [19]:
chunk_pred = [ent for ent in ner_predicts if ent[1] != 'other']
print(chunk_pred)

[['Earth System Models', 'model'], ['ESMs', 'model'], ['ESMs', 'model'], ['tropics', 'location'], ['USA', 'location'], ['Europe', 'location'], ['China', 'location'], ['low-income countries', 'location']]


In [20]:
chunk_pred

[['Earth System Models', 'model'],
 ['ESMs', 'model'],
 ['ESMs', 'model'],
 ['tropics', 'location'],
 ['USA', 'location'],
 ['Europe', 'location'],
 ['China', 'location'],
 ['low-income countries', 'location']]

## CrossNER-AI

基础Workflow:

Workflow to improve the LLMs NER performance
- Mention Detection
    - CoT
    - Generation Format:
        - Has Negative: Directly generate entity mention span
        - No Negative: Using special tag, <e>Mention Span<e>
- Entity Typing

In [4]:
zero_shot_data_processing_steps = [
    """You are a mention detection assistant. Your task is to identify named entities in a given text and wrap each entity mention with <entity> and </entity> tags. 

    Instructions:
    1. Return the *exact same text*, but with each named entity enclosed in <entity>...</entity> tags.
    2. Named entities are from AI domain, including 'algorithm', 'conference', 'country', 'field', 'location', 'metrics', 'miscellaneous', 'organization', 'person', 'product', 'programming language', 'researcher', 'task', 'university'.
    3. If there are multiple words in an entity (e.g., “Deep Neural Network”), wrap the entire phrase: <entity>Deep Neural Network</entity>.
    4. Keep the rest of the text unchanged.
    5. Do not add any extra commentary or formatting. Only return the modified text.
    6. If there are no entities, simply return the original text without any tags.""",
    
    """You are an entity typing assistant. You will receive text where certain substrings are wrapped in <entity>...</entity> tags. These substrings are entity mentions. 

    Your task:
    1. For each mention wrapped in <entity>...</entity>, determine the most suitable entity type from the following list:
    - algorithm
    - conference
    - country
    - field
    - location
    - metrics
    - miscellaneous
    - organization
    - person
    - product
    - programming language
    - researcher
    - task
    - university
    - other  (use this if the mention does not fit any of the above)

    2. Convert each <entity>...</entity> tag into <entity type="TYPE">...</entity>, where TYPE is one of the listed options. 
    - Example: If the mention is "Deep Neural Network" and you decide it’s an 'algorithm', change <entity>Deep Neural Network</entity> to <entity type="algorithm">Deep Neural Network</entity>.

    3. Leave all other text (outside the entity tags) exactly the same.

    4. If a mention is ambiguous or does not clearly fit any of the predefined types, classify it as "other".

    5. Output only the modified text, with the <entity> tags updated to include the type. Do not add any extra commentary, code fences, or explanation."""
]

In [5]:
from tqdm import trange
data = load_json("/home/tuo96248/projects/NER-DAug/datasets/crossner/ai_test.json")

for i in trange(len(data)):
    str_words = data[i]['str_words']
    str_tags = data[i]['tags_ner']
    ner_labels = get_ners(str_words, str_tags)
    input_sent = " ".join(str_words)
    # ---- LLM predict ----
    output = chain(input_sent, zero_shot_data_processing_steps)
    ner_predicts = extract_entities(output[-1])
    break

  0%|          | 0/431 [00:00<?, ?it/s]

Typical generative model approaches include naive Bayes classifier s , Gaussian mixture model s , variational autoencoders and others .
Typical generative model approaches include <entity>naive Bayes classifier</entity>s , <entity>Gaussian mixture model</entity>s , <entity>variational autoencoders</entity> and others .


  0%|          | 0/431 [00:04<?, ?it/s]


In [6]:
output

['Typical generative model approaches include naive Bayes classifier s , Gaussian mixture model s , variational autoencoders and others .',
 'Typical generative model approaches include <entity>naive Bayes classifier</entity>s , <entity>Gaussian mixture model</entity>s , <entity>variational autoencoders</entity> and others .',
 'Typical generative model approaches include <entity type="algorithm">naive Bayes classifier</entity>s , <entity type="algorithm">Gaussian mixture model</entity>s , <entity type="algorithm">variational autoencoders</entity> and others .']

In [7]:
output[-1]

'Typical generative model approaches include <entity type="algorithm">naive Bayes classifier</entity>s , <entity type="algorithm">Gaussian mixture model</entity>s , <entity type="algorithm">variational autoencoders</entity> and others .'

In [8]:
ner_predicts

[['naive Bayes classifier', 'algorithm'],
 ['Gaussian mixture model', 'algorithm'],
 ['variational autoencoders', 'algorithm']]

In [13]:
# Svae
save_json(data, "/home/tuo96248/projects/NER-DAug/output/workflow/ai_test.json")

In [43]:
def safe_div(num, denom):
    if denom > 0:
        return num / denom
    else:
        return 0


def compute_f1(predicted, gold, matched):
    # F1 score.
    precision = safe_div(matched, predicted)
    recall = safe_div(matched, gold)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return dict(precision=precision, recall=recall, f1=f1)


def evaluate_sent(gt_ner, pred_ner,counts):
    # correct_ner = set()
    # Entities.
    counts["ner_gold"] += len(gt_ner)
    counts["ner_predicted"] += len(pred_ner)
    for prediction in pred_ner:
        if any([prediction == actual for actual in gt_ner]):
            counts["ner_matched"] += 1
            # correct_ner.add(prediction[0])
    return counts


In [47]:
from collections import Counter
counts = Counter()
# read json
data = load_json("/home/tuo96248/projects/NER-DAug/output/workflow/ai_test.json")
mapping = AI_CLASSS
for i in range(len(data)):
    ner_labels = data[i]['ner_labels']
    ner_predicts = data[i]['ner_predicts']
    # update ner_labels
    for idx in range(len(ner_labels)):
        ner_labels[idx][1] = mapping[ner_labels[idx][1]]
    counts = evaluate_sent(ner_labels, ner_predicts, counts)
scores_ner = compute_f1(
            counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
print(scores_ner)

{'precision': 0.597623089983022, 'recall': 0.5837479270315091, 'f1': 0.5906040268456376}


{'precision': 0.597623089983022, 'recall': 0.5837479270315091, 'f1': 0.5906040268456376}


In [42]:
ner_labels

[['FrameNet', 'product'],
 ['question answering', 'task'],
 ['paraphrasing', 'task'],
 ['recognizing textual entailment', 'task'],
 ['information extraction', 'task'],
 ['Semantic Role Labeling', 'task']]

In [14]:
AI_CLASSS

{'algorithm': 'algorithm',
 'conference': 'conference',
 'country': 'country',
 'field': 'field',
 'location': 'location',
 'metrics': 'metrics',
 'misc': 'miscellaneous',
 'organisation': 'organization',
 'person': 'person',
 'product': 'product',
 'programlang': 'programming language',
 'researcher': 'researcher',
 'task': 'task',
 'university': 'university'}

In [7]:
sample = data[0]
sentence = ' '.join(sample['str_words'])
str_words = sample['str_words']
str_tags = sample['tags_ner']
ner_labels = get_ners(str_words, str_tags)

In [8]:
ner_labels

[['naive Bayes classifier', 'algorithm'],
 ['Gaussian mixture model', 'algorithm'],
 ['variational autoencoders', 'algorithm']]

In [39]:
sentence

'Typical generative model approaches include naive Bayes classifier s , Gaussian mixture model s , variational autoencoders and others .'

In [30]:
data[0].keys()

dict_keys(['str_words', 'tags_ner', 'tags_esi', 'tags_net', 'sent_id'])

In [40]:
res = chain(sentence, zero_shot_data_processing_steps)

'Typical generative model approaches include <e type="algorithm">naive Bayes classifier</e>s, <e type="algorithm">Gaussian mixture model</e>s, <e type="algorithm">variational autoencoders</e> and others.'

In [44]:
ents = extract_entities(res[-1])

In [45]:
ents

[['naive Bayes classifier', 'algorithm'],
 ['Gaussian mixture model', 'algorithm'],
 ['variational autoencoders', 'algorithm']]